## Concatenate GO & Louvain Results

## Purpose: 

1. Take all the `GO` results produced from `./2_louvain_clustering_submission.py`. 
2. Subset to results where `p_fdr_benjamini_hochberg` <0.05
3. Save metadata about that `GO` result file
4. Concatenate all `Pandas DataFrames`. 
5. Output data as `gzip'd TSV` file. 
6. Remove all the `SLURM` and script intermediary files produced from `./2_louvain_clustering_submission.py` 

Separately, store the results from all `Louvain clusterings` in `dictionary` and then `pickle`.

In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pandas as pd
pd.set_option('display.max_rows', 1000000)
pd.set_option('display.max_columns', 1000000)
pd.set_option('display.max_colwidth', 10000000)

import glob, os, shutil, pickle

## Load Data

In [2]:
parameter_combinations = [file.split("/")[-1].split(".")[0] for file in sorted(glob.glob("/home/jve4pt/RBFOX2-Mouse-Embryonic-Heart-Analysis/outputs/pickled_networks/differential_expression_tf_networks/*", recursive=True))]

parameter_combinations

['upstream_2500_score_900',
 'upstream_2500_score_950',
 'upstream_2500_score_990',
 'upstream_5000_score_900',
 'upstream_5000_score_950',
 'upstream_5000_score_990',
 'upstream_750_score_900',
 'upstream_750_score_950',
 'upstream_750_score_990']

## Concatenation Algorithm

In [ ]:
# # ONLY FOR USE IF DATA CONCATENATION IS LARGE

# for parameter_combination in parameter_combinations:
    
#     output_file = "/scratch/jve4pt/{}_output.txt".format(parameter_combination)
#     error_file = "/scratch/jve4pt/{}_error.txt".format(parameter_combination)

#     os.system("sbatch --output='{}' --error='{}' --partition=largemem --mem=10GB -N1 -n1 --wrap='/apps/software/standard/core/anaconda/2020.11-py3.8/bin/python /sfs/qumulo/qhome/jve4pt/RBFOX2-Mouse-Embryonic-Heart-Analysis/prior_data/tf_analysis/7_louvain_clustering_and_gene_ontology_analysis/code/3_concatenate_large_GO_data.py {}'".format(output_file, error_file, parameter_combination))

In [3]:
# ONLY FOR /scratch/jve4pt/

# list with all the Gene Ontology analysis results in one
meta_GO_df = []

"""
for GO results for all possible parameter combinations, iterations, and clusters, 
retrieve them via `glob`, subset to FDR p < 0.05, 
save info about parameter combination, iteration and cluster
and finally concatenate all these together

# OPTIONAL: filtering GO terms by keywords

"""



for parameter_combination in parameter_combinations:
    
    parameter_combination
    
                
    tmp_files = glob.glob(
        "/scratch/jve4pt/*{}*.tsv.gz".format(parameter_combination)
    )

    if len(tmp_files)==0: 
        print("{} {}".format(parameter_combination))

    for file in tmp_files: 

        df = pd.read_csv(file , sep="\t")

        df = df[(df["p_uncorrected"]<0.05) & (df["NS"] =="BP")]

#             # subset to cardiac relevant tissue 
#             df =df[(df["name"].str.contains("blood")) | (df["name"].str.contains("heart")) | (df["name"].str.contains("vascul")) | (df["name"].str.contains("circul")) | (df["name"].str.contains("cardiac"))]

        if df.index.size>0:


            meta_GO_df.append(df)


meta_GO_df = pd.concat(meta_GO_df)

'\nfor GO results for all possible parameter combinations, iterations, and clusters, \nretrieve them via `glob`, subset to FDR p < 0.05, \nsave info about parameter combination, iteration and cluster\nand finally concatenate all these together\n\n# OPTIONAL: filtering GO terms by keywords\n\n'

'upstream_2500_score_900'

'upstream_2500_score_950'

'upstream_2500_score_990'

'upstream_5000_score_900'

'upstream_5000_score_950'

'upstream_5000_score_990'

'upstream_750_score_900'

'upstream_750_score_950'

'upstream_750_score_990'

In [ ]:
# # dataframe with all the Gene Ontology analysis results in one
# meta_GO_df = pd.DataFrame()

# """
# for GO results for all possible parameter combinations, iterations, and clusters, 
# retrieve them via `glob`, subset to FDR p < 0.05, 
# save info about parameter combination, iteration and cluster
# and finally concatenate all these together

# # OPTIONAL: filtering GO terms by keywords

# """



# for parameter_combination in parameter_combinations:
    
#     parameter_combination
    
#     for iteration in range(1000): 
                
#         tmp_files = glob.glob(
#             "/scratch/jve4pt/gene_ontology/differential_expression_networks/{}-iter_{}/*.tsv".format(parameter_combination, iteration)
#         )
        
#         if len(tmp_files)==0: 
#             print("{} {}".format(parameter_combination, iteration))
            
#         for file in tmp_files: 
            
#             cluster = file.split("-")[-1].split(".")[0]
                        
#             df = pd.read_csv(file , sep="\t")
            
#             df = df[df["p_uncorrected"]<0.05].copy()
                        
# #             # subset to cardiac relevant tissue 
# #             df =df[(df["name"].str.contains("blood")) | (df["name"].str.contains("heart")) | (df["name"].str.contains("vascul")) | (df["name"].str.contains("circul")) | (df["name"].str.contains("cardiac"))]
            
#             if df.index.size>0:
                
#                 df["Parameter_Combination"] = parameter_combination
#                 df["Iteration"] = iteration 
#                 df["Cluster"] = cluster
            
#                 meta_GO_df = pd.concat([meta_GO_df, df])



In [4]:
meta_GO_df.index.size

meta_GO_df.head()

22664465

,# GO,NS,enrichment,name,ratio_in_study,ratio_in_pop,p_uncorrected,depth,study_count,p_fdr_bh,study_items,Parameter_Combination,Iteration,Cluster
0,GO:0014860,BP,e,neurotransmitter secretion involved in regulation of skeletal muscle contraction,1/19,1/21965,0.000865,9,1,1.0,nr4a1,upstream_2500_score_900,0,cluster_6
1,GO:0018900,BP,e,dichloromethane metabolic process,1/19,1/21965,0.000865,5,1,1.0,gstt1,upstream_2500_score_900,0,cluster_6
2,GO:1903954,BP,e,positive regulation of voltage-gated potassium channel activity involved in atrial cardiac muscle cell action potential repolarization,1/19,1/21965,0.000865,12,1,1.0,rnf207,upstream_2500_score_900,0,cluster_6
3,GO:1903762,BP,e,positive regulation of voltage-gated potassium channel activity involved in ventricular cardiac muscle cell action potential repolarization,1/19,1/21965,0.000865,12,1,1.0,rnf207,upstream_2500_score_900,0,cluster_6
4,GO:1900542,BP,e,regulation of purine nucleotide metabolic process,1/19,1/21965,0.000865,8,1,1.0,lacc1,upstream_2500_score_900,0,cluster_6


## Output `DataFrame` in `gzip` Format 

In [ ]:
meta_GO_df.to_csv(
    "../../../../outputs/gene_ontology_analysis/differential_expression_networks/all_GO_analyses_p_uncorrected_0.05.tsv.gz", 
    sep="\t", 
    compression='gzip', 
    index=False
)

Make sure that file is properly formatted

In [ ]:
tmp = pd.read_csv(
    "../../../../outputs/gene_ontology_analysis/differential_expression_networks/all_GO_analyses_p_uncorrected_0.05.tsv.gz", 
    sep="\t",
    compression='gzip',
)

tmp.index.size
tmp.head()

## Remove Original Output from Script + `SLURM`

In [ ]:
# if os.path.exists("../../../../outputs/gene_ontology_analysis/differential_expression_networks/output/"): 
#     shutil.rmtree("../../../../outputs/gene_ontology_analysis/differential_expression_networks/output/")
    
# if os.path.exists("../../../../outputs/gene_ontology_analysis/differential_expression_networks/SLURM/"):    
#     shutil.rmtree("../../../../outputs/gene_ontology_analysis/differential_expression_networks/SLURM/")

## Create Dict of Louvain Results and Pickle

In [ ]:
louvain_results = {}

# load all the pickled louvain community clustering results 
# into a dict such that it's organized by parameter combination and iteration

for parameter_combination in parameter_combinations:
        
    louvain_results[parameter_combination]={}
    
    for iteration in range(1000): 
        louvain_results[parameter_combination][iteration] = pickle.load(open("/scratch/jve4pt/louvain_clustering/differential_expression_networks/{}/{}-iter_{}.pkl".format(parameter_combination, parameter_combination, iteration), 'rb'))


In [ ]:
pickle.dump(louvain_results, open("../../../../outputs/louvain_clustering/all_louvain_clusterings.pkl", 'wb'))

In [ ]:
tmp_test = pickle.load(open("../../../../outputs/louvain_clustering/all_louvain_clusterings.pkl", 'rb'))

In [ ]:
tmp_test["upstream_2500_score_900"][1][0]

## Remove Extraneous Clustering Data Folders

In [ ]:
# folders = glob.glob("../../../../outputs/louvain_clustering/upstream_*")
# folders

In [ ]:
# for folder in folders: 
#     shutil.rmtree(folder)

## Remove this Later

In [8]:
baseline_append = pd.DataFrame()

for parameter_combination in parameter_combinations:
    
    parameter_combination
    
    for iteration in [1000]: 
                
        tmp_files = glob.glob(
            "/scratch/jve4pt/gene_ontology/differential_expression_networks/{}-iter_{}/*.tsv".format(parameter_combination, iteration)
        )
        
        if len(tmp_files)==0: 
            print("{} {}".format(parameter_combination, iteration))
            
        for file in tmp_files: 
            
            cluster = file.split("-")[-1].split(".")[0]
                        
            df = pd.read_csv(file , sep="\t")
            
            df = df[(df["p_uncorrected"]<0.05) & (df["NS"] == "BP")]
                        
#             # subset to cardiac relevant tissue 
#             df =df[(df["name"].str.contains("blood")) | (df["name"].str.contains("heart")) | (df["name"].str.contains("vascul")) | (df["name"].str.contains("circul")) | (df["name"].str.contains("cardiac"))]
            
            if df.index.size>0:
                
                df["Parameter_Combination"] = parameter_combination
                df["Iteration"] = iteration 
                df["Cluster"] = cluster
            
                baseline_append = pd.concat([baseline_append, df])

'upstream_2500_score_900'

'upstream_2500_score_950'

'upstream_2500_score_990'

'upstream_5000_score_900'

'upstream_5000_score_950'

'upstream_5000_score_990'

'upstream_750_score_900'

'upstream_750_score_950'

'upstream_750_score_990'

In [9]:
baseline_append.head()

,# GO,NS,enrichment,name,ratio_in_study,ratio_in_pop,p_uncorrected,depth,study_count,p_fdr_bh,study_items,Parameter_Combination,Iteration,Cluster
0,GO:1905703,BP,e,negative regulation of inhibitory synapse assembly,1/13,1/21965,0.000592,8,1,1.0,cbln1,upstream_2500_score_900,1000,cluster_1
1,GO:0070194,BP,e,synaptonemal complex disassembly,1/13,2/21965,0.001183,7,1,1.0,plk1,upstream_2500_score_900,1000,cluster_1
2,GO:1904776,BP,e,regulation of protein localization to cell cortex,1/13,2/21965,0.001183,7,1,1.0,plk1,upstream_2500_score_900,1000,cluster_1
3,GO:2001213,BP,e,negative regulation of vasculogenesis,1/13,2/21965,0.001183,6,1,1.0,xdh,upstream_2500_score_900,1000,cluster_1
4,GO:0010044,BP,e,response to aluminum ion,1/13,2/21965,0.001183,5,1,1.0,xdh,upstream_2500_score_900,1000,cluster_1


In [11]:
baseline_append.to_csv(
    "../../../../outputs/gene_ontology_analysis/differential_expression_networks/baseline_networks_GO_p_uncorrected_0.05.tsv.gz", 
    compression="gzip", 
    index=False, 
    sep="\t"
)

In [30]:
tmp = pd.DataFrame(meta_GO_df[meta_GO_df["p_fdr_bh"]<0.05].groupby(["name"]).median())
tmp.head()

22664465

In [28]:
tmp = pd.DataFrame(baseline_append.groupby(["name"])["p_fdr_bh"].median())

tmp.head()

,p_fdr_bh
name,
'de novo' L-methionine biosynthetic process,0.628771
"1-phosphatidyl-1D-myo-inositol 4,5-bisphosphate biosynthetic process",0.795930
10-formyltetrahydrofolate catabolic process,1.000000
3'-UTR-mediated mRNA stabilization,1.000000
9-cis-retinoic acid biosynthetic process,1.000000


In [27]:
tmp.sort_values(["p_uncorrected"], ascending=True).head()

,p_uncorrected
name,
fibroblast growth factor receptor signaling pathway involved in positive regulation of cell proliferation in bone marrow,0.000228
lateral sprouting from an epithelium,0.000228
coronal suture morphogenesis,0.000228
fibroblast growth factor receptor signaling pathway involved in negative regulation of apoptotic process in bone marrow cell,0.000228
fibroblast growth factor receptor signaling pathway involved in hemopoiesis,0.000228


In [32]:
meta_GO_df = meta_GO_df[meta_GO_df["p_fdr_bh"]<0.05]

meta_GO_df = pd.DataFrame(meta_GO_df["name"].value_counts())

In [34]:
meta_GO_df.head(n=100)

,name
cholesterol metabolic process,4080
multicellular organism development,2835
cholesterol homeostasis,2522
steroid metabolic process,2403
lipoprotein metabolic process,2156
positive regulation of TRAIL-activated apoptotic signaling pathway,2082
sterol biosynthetic process,1987
canonical Wnt signaling pathway,1943
gene expression,1928
Wnt signaling pathway,1907
